In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from pydantic import Field, BaseModel
from typing import Literal

In [7]:
load_dotenv()

model = ChatOpenAI()

parser_1 = StrOutputParser()

In [8]:
class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser_2 = PydanticOutputParser(pydantic_object=Feedback)

In [13]:
prompt_1 = PromptTemplate(
    template="Read the given feedback and classify its sentiment as postive or negative \n {feedback} \n {format_instruction}",
    input_variables=['feedback'],
    partial_variables={'format_instruction': parser_2.get_format_instructions()}
)

classifier_chain = prompt_1 | model | parser_2

In [17]:
prompt_2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt_3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', prompt_2 | model | parser_1),
    (lambda x: x.sentiment == 'negative', prompt_3 | model | parser_1),
    RunnableLambda(lambda x: 'could not find sentiment')
)

In [19]:
chain = classifier_chain | branch_chain
result = chain.invoke({'feedback': 'This is a wonderful phone.'})
print(result)

Thank you so much for your kind words and encouragement! It means a lot to me and I'm grateful for your support. I will continue to work hard and strive for excellence.


In [20]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
